# Aprendizado Não Supervisionado — Módulo 1
### Por que importa — e os cinco erros que quase todo mundo comete

**IA Aplicada · UNISINOS · Pós-Graduação**

---

Este notebook é o laboratório da aula. Cada um dos cinco erros aparece aqui em
código executável, com o número que foi para o slide reproduzido a partir de dados
sintéticos e `random_state` fixo — você roda de novo e obtém o mesmo valor.

**Como usar:** `Ambiente de execução → Executar tudo`. Tempo total ≈ 3 min.

| Bloco | Erro | O que a demonstração mostra |
|---|---|---|
| 1 | Rodar sem padronizar | 99,93% da distância vem de um atributo; ARI 0,54 → 0,96 |
| 2 | Deixar o gráfico escolher *k* | silhueta aponta 2; a resposta certa é 4 |
| 3 | Reportar acurácia com alvo raro | acurácia 97,1% para um modelo que nunca dispara |
| 4 | Confundir co-ocorrência com relação | confiança 0,92 e lift 1,49 *vs* lift 5,70 |
| 5 | Usar informação que não existia | janela centrada: 0 alarmes numa falha real |

> Os dados são sintéticos **de propósito**: sem gabarito construído por nós, seria
> impossível provar que o pipeline errou — que é exatamente o problema do módulo.

## 0 · Setup

Bibliotecas, paleta institucional e helpers de animação.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (adjusted_rand_score, silhouette_score, silhouette_samples,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, average_precision_score)

RS = 42                      # random_state global — tudo aqui é reprodutível
np.random.seed(RS)

# paleta institucional UNISINOS
AZUL, AZUL_D, VERM = "#253785", "#214099", "#A32330"
CINZA, CLARO, LINHA = "#3F4757", "#EFF2FA", "#D3DAEB"
PAL4 = ["#253785", "#4C6BC4", "#A32330", "#D98C2B"]

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelcolor": CINZA, "axes.edgecolor": LINHA, "axes.titlecolor": AZUL,
    "xtick.color": CINZA, "ytick.color": CINZA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LINHA, "grid.linewidth": 0.8,
    "legend.frameon": False,
    "animation.embed_limit": 80,          # MB — as animações são embutidas como JS
})

def mostrar(anim):
    """Renderiza a animação como player HTML (funciona no Colab sem ffmpeg)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml(default_mode="loop"))

print("Setup concluído · numpy", np.__version__, "· pandas", pd.__version__)

## 1 · Os dados: base de clientes com gabarito conhecido

Quatro segmentos reais (`A`, `B`, `C`, `D`) definidos por **comportamento** —
frequência de compra e recência. O atributo `gasto_mensal` está em reais e,
por isso, numa escala mil vezes maior que os demais.

A estrutura é deliberada: `A` e `B` estão próximos entre si, assim como `C` e `D`,
e os dois pares estão longe um do outro. Guarde isso — é a armadilha do Bloco 3.

In [ ]:
COLS = ["gasto_mensal", "tempo_sessao_seg", "frequencia_mes",
        "recencia_dias", "tickets_suporte", "nps"]
NOMES = {0: "A", 1: "B", 2: "C", 3: "D"}

def gerar_clientes(n_por_grupo=250, seed=RS):
    """4 segmentos; a separação real está em frequencia_mes e recencia_dias."""
    rng = np.random.default_rng(seed)
    # posição de cada segmento no espaço comportamental (z-score)
    z = {"A": (-1.6, -1.5), "B": (-1.6, 1.5), "C": (1.6, -1.5), "D": (1.6, 1.5)}
    # gasto: separa bem os PARES (A,B) x (C,D) e mal os grupos dentro do par
    gasto = {"A": 2050, "B": 3300, "C": 8050, "D": 9300}
    blocos, y = [], []
    for i, (g, (z1, z2)) in enumerate(z.items()):
        f = rng.normal(z1, 0.70, n_por_grupo)
        r = rng.normal(z2, 0.70, n_por_grupo)
        blocos.append(np.column_stack([
            rng.normal(gasto[g], 820, n_por_grupo),   # R$ por mês
            rng.normal(240, 80, n_por_grupo),         # segundos por sessão
            6 + 1.2 * f,                              # compras por mês
            30 + 6 * r,                               # dias desde a última compra
            rng.normal(3.0, 0.8, n_por_grupo),        # tickets de suporte
            rng.normal(7.5, 1.2, n_por_grupo),        # NPS
        ]))
        y += [i] * n_por_grupo
    return np.vstack(blocos), np.array(y)

X, y = gerar_clientes()
df = pd.DataFrame(X, columns=COLS).assign(segmento=[NOMES[i] for i in y])
display(df.groupby("segmento")[COLS].mean().round(1))
print("shape:", X.shape)

In [ ]:
# amplitude de cada atributo — a raiz do problema, visível antes de qualquer modelo
fig, ax = plt.subplots(figsize=(8, 3.4))
desv = df[COLS].std().sort_values()
ax.barh(desv.index, desv.values, color=[VERM if v > 100 else AZUL for v in desv.values])
ax.set_xscale("log")
ax.set_xlabel("desvio-padrão do atributo (escala log)")
ax.set_title("Seis atributos, três ordens de grandeza de diferença")
for i, v in enumerate(desv.values):
    ax.text(v * 1.15, i, f"{v:,.1f}".replace(",", "."), va="center", color=CINZA, fontsize=10)
ax.set_xlim(0.5, desv.max() * 8)
plt.tight_layout(); plt.show()

---
## 2 · Erro 01 — rodar o algoritmo sem padronizar

**O que parece:** você passa os seis atributos, o K-Means devolve quatro grupos,
nada acusa problema.

**O que é:** distância euclidiana soma quadrados. Um atributo em milhares
esmaga cinco atributos em unidades — e o algoritmo não avisa.

In [ ]:
def fracao_da_distancia(X, cols, n_pares=6000, seed=RS):
    """Quanto cada atributo contribui, em média, para a distância² entre dois clientes."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=(n_pares, 2))
    d2 = (X[idx[:, 0]] - X[idx[:, 1]]) ** 2
    return pd.Series(d2.sum(axis=0) / d2.sum(), index=cols)

frac_bruto = fracao_da_distancia(X, COLS)
Xs = StandardScaler().fit_transform(X)          # z-score: média 0, desvio 1
frac_pad = fracao_da_distancia(Xs, COLS)

print(f"Antes de padronizar : {frac_bruto.max():.4%} da distância vem de "
      f"'{frac_bruto.idxmax()}'")
print(f"Depois de padronizar: {frac_pad.max():.4%} (maior contribuição individual)")
pd.DataFrame({"bruto": frac_bruto, "padronizado": frac_pad}).style.format("{:.4%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, serie, titulo in zip(axes, [frac_bruto, frac_pad],
                             ["Dados brutos", "Após StandardScaler"]):
    cores = [VERM if v > 0.5 else AZUL for v in serie.values]
    ax.bar(range(len(serie)), serie.values * 100, color=cores)
    ax.set_xticks(range(len(serie)))
    ax.set_xticklabels(serie.index, rotation=35, ha="right", fontsize=9)
    ax.set_ylabel("% da distância²"); ax.set_ylim(0, 105); ax.set_title(titulo)
axes[0].annotate(f"{frac_bruto.max():.2%}", xy=(0, 100), xytext=(0.7, 78),
                 color=VERM, fontweight="bold", fontsize=13)
plt.suptitle("Quem realmente define quem é parecido com quem", color=AZUL, y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# o efeito prático: mesma base, mesmo algoritmo, mesmo k — só muda a preparação
km_bruto = KMeans(4, n_init=10, random_state=RS).fit(X)
km_pad   = KMeans(4, n_init=10, random_state=RS).fit(Xs)

ari_bruto = adjusted_rand_score(y, km_bruto.labels_)
ari_pad   = adjusted_rand_score(y, km_pad.labels_)

print(f"ARI sem padronizar : {ari_bruto:.3f}")
print(f"ARI com padronizar : {ari_pad:.3f}")
print(f"ganho: +{(ari_pad - ari_bruto) / ari_bruto:.0%} — só com uma linha de pré-processamento")

fig, ax = plt.subplots(figsize=(6.4, 3.2))
barras = ax.bar(["sem padronizar", "com padronizar"], [ari_bruto, ari_pad],
                color=[VERM, AZUL], width=0.55)
ax.bar_label(barras, fmt="%.3f", padding=4, color=CINZA, fontweight="bold")
ax.set_ylim(0, 1.1); ax.set_ylabel("ARI (1,0 = recuperou o gabarito)")
ax.set_title("Adjusted Rand Index contra o gabarito conhecido")
plt.tight_layout(); plt.show()

### Animação 1 — o que a padronização faz com o espaço

Interpolação contínua entre o espaço bruto e o espaço padronizado, projetada nos
dois atributos que **de fato** definem os segmentos. As cores são o gabarito.

Repare no eixo: no início os pontos estão espremidos numa faixa (a distância só
enxerga o gasto); no fim os quatro segmentos aparecem.

In [ ]:
def animar_padronizacao(X, y, i_gasto=0, i_freq=2, n_frames=48):
    esc = StandardScaler().fit(X)
    A = X[:, [i_gasto, i_freq]].copy()
    B = esc.transform(X)[:, [i_gasto, i_freq]]
    # normaliza o bruto só para caber na mesma tela (a FORMA é o que importa)
    A = (A - A.mean(0)) / np.array([A[:, 0].std(), A[:, 0].std()])

    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    pts = ax.scatter(A[:, 0], A[:, 1], s=12, c=[PAL4[i] for i in y], alpha=0.75,
                     edgecolors="none")
    ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
    ax.set_xlabel("gasto_mensal"); ax.set_ylabel("frequencia_mes")
    titulo = ax.set_title("")

    def frame(k):
        t = k / (n_frames - 1)
        s = t * t * (3 - 2 * t)                      # smoothstep
        P = (1 - s) * A + s * B
        pts.set_offsets(P)
        titulo.set_text(f"escala original  →  padronizada   ({s:5.0%})")
        return pts, titulo

    return animation.FuncAnimation(fig, frame, frames=n_frames, interval=70, blit=False)

mostrar(animar_padronizacao(X, y))

> **Como evitar:** padronize antes de qualquer método baseado em distância — e
> depois confira se todo atributo que ganhou peso igual **merece** peso igual.
> Escala é decisão de modelagem, não faxina de dados.

---
## 3 · Erro 02 — deixar o gráfico escolher o número de grupos

**O que parece:** rodar o método do cotovelo e a silhueta, pegar o melhor valor,
seguir em frente. Parece objetivo.

**O que é:** a métrica premia grupos geometricamente limpos, não grupos úteis.

In [ ]:
ks = range(2, 11)
inercia, silhueta, ari_k = [], [], []
for k in ks:
    m = KMeans(k, n_init=10, random_state=RS).fit(Xs)
    inercia.append(m.inertia_)
    silhueta.append(silhouette_score(Xs, m.labels_))
    ari_k.append(adjusted_rand_score(y, m.labels_))   # só existe porque temos gabarito

tab = pd.DataFrame({"k": list(ks), "inercia": np.round(inercia, 0),
                    "silhueta": np.round(silhueta, 3), "ARI_real": np.round(ari_k, 3)})
k_silhueta = tab.loc[tab.silhueta.idxmax(), "k"]
k_verdade  = tab.loc[tab.ARI_real.idxmax(), "k"]
display(tab)
print(f"A silhueta escolheria k = {k_silhueta}.  O gabarito diz k = {k_verdade}.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
for ax, val, nome in zip(axes, [inercia, silhueta, ari_k],
                         ["Inércia (cotovelo)", "Silhueta", "ARI real (só com gabarito)"]):
    ax.plot(list(ks), val, "o-", color=AZUL, lw=2, ms=6)
    ax.set_xlabel("k"); ax.set_title(nome)
axes[1].plot(k_silhueta, max(silhueta), "o", ms=14, mfc="none", mec=VERM, mew=2.5)
axes[1].annotate("a métrica escolhe 2", xy=(k_silhueta, max(silhueta)),
                 xytext=(4.2, max(silhueta) * 0.93), color=VERM, fontweight="bold",
                 arrowprops=dict(arrowstyle="->", color=VERM))
axes[2].plot(k_verdade, max(ari_k), "o", ms=14, mfc="none", mec=AZUL, mew=2.5)
axes[2].annotate("a verdade é 4", xy=(k_verdade, max(ari_k)), xytext=(5.4, max(ari_k) * 0.8),
                 color=AZUL, fontweight="bold", arrowprops=dict(arrowstyle="->", color=AZUL))
plt.tight_layout(); plt.show()

In [ ]:
# por que a silhueta erra: com k=2 ela encontra dois blocos geometricamente perfeitos
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, k in zip(axes, [2, 4]):
    lab = KMeans(k, n_init=10, random_state=RS).fit_predict(Xs)
    ax.scatter(Xs[:, 2], Xs[:, 3], s=12, c=[PAL4[i % 4] for i in lab], alpha=0.75,
               edgecolors="none")
    ax.set_title(f"k = {k}   ·   silhueta {silhouette_score(Xs, lab):.3f}   ·   "
                 f"ARI {adjusted_rand_score(y, lab):.3f}")
    ax.set_xlabel("frequencia_mes (padronizado)"); ax.set_ylabel("recencia_dias (padronizado)")
plt.suptitle("A solução mais 'limpa' não é a solução certa", color=AZUL, y=1.02)
plt.tight_layout(); plt.show()

### Animação 2 — o K-Means iterando

Cada frame é uma iteração de Lloyd: atribuir pontos ao centróide mais próximo,
recalcular centróides. O algoritmo converge — mas converge para o *k* que você deu a ele.

In [ ]:
def animar_kmeans(Xp, k=4, n_iter=12, seed=7):
    rng = np.random.default_rng(seed)
    P = Xp[:, [2, 3]]
    cent = P[rng.choice(len(P), k, replace=False)].copy()     # inicialização aleatória

    hist = []
    for _ in range(n_iter):
        d = ((P[:, None, :] - cent[None, :, :]) ** 2).sum(-1)
        lab = d.argmin(1)
        hist.append((lab.copy(), cent.copy()))
        for j in range(k):
            if (lab == j).any():
                cent[j] = P[lab == j].mean(0)

    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    pts = ax.scatter(P[:, 0], P[:, 1], s=12, alpha=0.7, edgecolors="none")
    cen = ax.scatter([], [], s=260, marker="X", c=VERM, edgecolors="white", linewidths=1.5,
                     zorder=3)
    ax.set_xlabel("frequencia_mes (padronizado)"); ax.set_ylabel("recencia_dias (padronizado)")
    titulo = ax.set_title("")

    def frame(i):
        lab, c = hist[i]
        pts.set_color([PAL4[j % 4] for j in lab])
        cen.set_offsets(c)
        titulo.set_text(f"iteração {i + 1} de {len(hist)}   ·   k = {k} (fixado por você)")
        return pts, cen, titulo

    return animation.FuncAnimation(fig, frame, frames=len(hist), interval=520, blit=False)

mostrar(animar_kmeans(Xs))

> **Como evitar:** use a métrica para descartar opções absurdas, não para escolher
> a final. O número de grupos sai da conversa entre o gráfico e a operação:
> quantos tratamentos distintos a empresa consegue de fato executar?

---
## 4 · Erro 03 — reportar acurácia quando o alvo é raro

Trocamos de domínio: detecção de intrusão. 6.000 conexões, **2,9% são ataque**.

In [ ]:
def gerar_trafego(n=6000, taxa=0.029, seed=RS):
    """Tráfego de rede sintético. 90% dos ataques são ruidosos, 10% são discretos."""
    rng = np.random.default_rng(seed)
    n_atk = int(n * taxa); n_ok = n - n_atk
    normal = np.column_stack([rng.normal(500, 120, n_ok), rng.normal(12, 3, n_ok),
                              rng.normal(0.30, 0.08, n_ok), rng.normal(3.0, 0.9, n_ok)])
    forte = int(n_atk * 0.90); sutil = n_atk - forte
    a1 = np.column_stack([rng.normal(1450, 360, forte), rng.normal(46, 12, forte),
                          rng.normal(0.80, 0.12, forte), rng.normal(8.8, 2.3, forte)])
    a2 = np.column_stack([rng.normal(820, 110, sutil), rng.normal(22, 4, sutil),
                          rng.normal(0.48, 0.07, sutil), rng.normal(5.0, 1.0, sutil)])
    Xt = np.vstack([normal, a1, a2])
    yt = np.r_[np.zeros(n_ok), np.ones(n_atk)]
    p = rng.permutation(n)
    return Xt[p], yt[p]

COLS_NET = ["bytes_por_fluxo", "pacotes_por_seg", "razao_syn", "portas_distintas"]
Xt, yt = gerar_trafego()
Xts = StandardScaler().fit_transform(Xt)

acc_burro = 1 - yt.mean()      # modelo que responde "não é ataque" para tudo
print(f"Taxa real de ataque : {yt.mean():.1%}")
print(f"Acurácia do modelo que NUNCA dispara alerta: {acc_burro:.1%}")

In [ ]:
linhas = []
for c in [0.029, 0.04, 0.05, 0.08, 0.12, 0.16, 0.20, 0.25, 0.30]:
    pred = (IsolationForest(contamination=c, n_estimators=250,
                            random_state=RS).fit_predict(Xts) == -1).astype(int)
    linhas.append({
        "contamination": c,
        "acuracia":  accuracy_score(yt, pred),
        "precisao":  precision_score(yt, pred, zero_division=0),
        "recall":    recall_score(yt, pred),
        "f1":        f1_score(yt, pred),
        "alertas_dia": int(pred.sum()),
    })
sweep = pd.DataFrame(linhas)
display(sweep.round(3))
print(f"A acurácia cai de {sweep.acuracia.iloc[0]:.3f} para {sweep.acuracia.iloc[-1]:.3f} "
      f"(-{sweep.acuracia.iloc[0]-sweep.acuracia.iloc[-1]:.2f}), "
      f"enquanto o F1 desaba de {sweep.f1.iloc[0]:.2f} para {sweep.f1.iloc[-1]:.2f}.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4))

ax1.plot(sweep.contamination, sweep.acuracia, "o-", color=CINZA, lw=2, label="acurácia")
ax1.plot(sweep.contamination, sweep.f1, "o-", color=VERM, lw=2.5, label="F1 (qualidade real)")
ax1.plot(sweep.contamination, sweep.precisao, "s--", color=AZUL, lw=1.6, label="precisão")
ax1.plot(sweep.contamination, sweep.recall, "^--", color="#4C6BC4", lw=1.6, label="recall")
ax1.axhline(acc_burro, color=CINZA, ls=":", lw=1.4)
ax1.text(0.30, acc_burro + 0.015, f"modelo que nunca alerta: {acc_burro:.1%}",
         ha="right", fontsize=9, color=CINZA)
ax1.set_xlabel("contamination"); ax1.set_ylabel("valor da métrica"); ax1.set_ylim(0, 1.06)
ax1.set_title("A acurácia mal se move; a qualidade real despenca"); ax1.legend(fontsize=9)

# matriz de confusão do cenário ruim
pred_ruim = (IsolationForest(contamination=0.20, n_estimators=250,
                             random_state=RS).fit_predict(Xts) == -1).astype(int)
cm = confusion_matrix(yt, pred_ruim)
ax2.imshow(cm, cmap="Blues"); ax2.grid(False)
ax2.set_xticks([0, 1]); ax2.set_xticklabels(["previu normal", "previu ataque"])
ax2.set_yticks([0, 1]); ax2.set_yticklabels(["é normal", "é ataque"])
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f"{cm[i, j]:,}".replace(",", "."), ha="center", va="center",
                 fontsize=15, fontweight="bold",
                 color="white" if cm[i, j] > cm.max() / 2 else AZUL)
ax2.set_title(f"contamination = 0,20 · acurácia {accuracy_score(yt, pred_ruim):.1%}")
plt.tight_layout(); plt.show()

n_alertas = f"{pred_ruim.sum():,}".replace(",", ".")
n_falsos  = f"{int(cm[0, 1]):,}".replace(",", ".")
print(f"Nesse cenário 'aprovado pela acurácia', a equipe receberia {n_alertas} alertas "
      f"por rodada — {n_falsos} deles falsos.")

> **Como evitar:** em problema desbalanceado, use precisão e recall. E lembre que o
> número de alertas por dia é um **orçamento da equipe**, não um parâmetro estatístico.

---
## 5 · Erro 04 — confundir co-ocorrência com relação útil

8.000 cestas de supermercado. A regra `manteiga → leite` tem confiança altíssima.
A pergunta que ninguém faz: **quanta gente já levava leite de qualquer jeito?**

In [ ]:
def gerar_cestas(n=8000, seed=RS):
    rng = np.random.default_rng(seed)
    cestas = []
    for _ in range(n):
        c = set()
        manteiga = rng.random() < 0.22
        if manteiga: c.add("manteiga")
        if rng.random() < (0.912 if manteiga else 0.537): c.add("leite")   # leite é popular
        esponja = rng.random() < 0.09
        if esponja: c.add("esponja")
        if rng.random() < (0.885 if esponja else 0.085): c.add("detergente")
        for item, p in [("pao", .45), ("cafe", .33), ("arroz", .28),
                        ("cerveja", .20), ("banana", .30)]:
            if rng.random() < p: c.add(item)
        cestas.append(c)
    return cestas

cestas = gerar_cestas()
itens = sorted({i for c in cestas for i in c})

def suporte(*items):
    s = set(items)
    return float(np.mean([s <= c for c in cestas]))

def regra(a, b):
    s_ab, s_a, s_b = suporte(a, b), suporte(a), suporte(b)
    conf = s_ab / s_a
    return {"regra": f"{a} → {b}", "suporte": s_ab, "confianca": conf,
            "suporte_consequente": s_b, "lift": conf / s_b}

pares = [("manteiga", "leite"), ("esponja", "detergente"), ("pao", "leite"),
         ("cafe", "leite"), ("cerveja", "banana"), ("arroz", "leite")]
regras = pd.DataFrame([regra(a, b) for a, b in pares]).sort_values("confianca",
                                                                  ascending=False)
display(regras.round(3))

r1 = regra("manteiga", "leite"); r2 = regra("esponja", "detergente")
print(f"{r1['regra']}: confiança {r1['confianca']:.0%}, mas lift {r1['lift']:.2f}")
print(f"{r2['regra']}: confiança {r2['confianca']:.0%} e lift {r2['lift']:.2f}  ← esta muda a decisão")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4))

pop = pd.Series({i: suporte(i) for i in itens}).sort_values()
ax1.barh(pop.index, pop.values, color=[VERM if i == "leite" else AZUL for i in pop.index])
ax1.set_xlabel("suporte (fração das cestas)")
ax1.set_title("Popularidade de cada item, isolado")
ax1.text(pop["leite"] - 0.02, list(pop.index).index("leite"),
         f"{pop['leite']:.0%}", ha="right", va="center", color="white", fontweight="bold")

cor = [VERM if l < 2 else AZUL for l in regras.lift]
ax2.scatter(regras.confianca, regras.lift, s=190, c=cor, alpha=0.85, edgecolors="white",
            linewidths=1.5, zorder=3)
ax2.axhline(1, color=CINZA, ls="--", lw=1.4)
ax2.text(0.02, 1.06, "lift = 1 · a regra não acrescenta nada", fontsize=9, color=CINZA,
         transform=ax2.get_yaxis_transform())
for _, r in regras.iterrows():
    ax2.annotate(r["regra"], (r.confianca, r.lift), textcoords="offset points",
                 xytext=(0, 14), ha="center", fontsize=9, color=CINZA)
ax2.set_xlabel("confiança"); ax2.set_ylabel("lift (ganho sobre o acaso)")
ax2.set_xlim(0, 1.05); ax2.set_ylim(0, regras.lift.max() * 1.35)
ax2.set_title("Confiança alta ≠ regra útil")
plt.tight_layout(); plt.show()

> **Como evitar:** sempre compare a regra contra a popularidade do item sozinho.
> Confiança alta com lift perto de 1 significa apenas que o produto é popular — e
> nenhuma das duas coisas prova causa: prova que aparecem juntos.

---
## 6 · Erro 05 — usar informação que ainda não existia

Um sensor com falha real e sustentada a partir de `t = 380`. Duas implementações da
**mesma** técnica (média móvel + 3,5 desvios): uma com janela centrada, outra causal.

In [ ]:
def gerar_sensor(n=600, t_falha=380, seed=RS):
    rng = np.random.default_rng(seed)
    s = 50 + 0.6 * np.sin(np.arange(n) / 38.0) + rng.normal(0, 0.55, n)
    s[t_falha:] += 4.2                       # degradação permanente do equipamento
    return pd.Series(s)

W, TH, T_FALHA = 41, 3.5, 380
s = gerar_sensor()

# ERRADO: janela centrada — usa 20 leituras do FUTURO para julgar o instante t
mu_c = s.rolling(W, center=True, min_periods=W).mean()
sd_c = s.rolling(W, center=True, min_periods=W).std()
z_c = (s - mu_c) / sd_c

# CERTO: janela causal — só o passado, e o próprio ponto fica de fora da estatística
mu_p = s.shift(1).rolling(W, min_periods=W).mean()
sd_p = s.shift(1).rolling(W, min_periods=W).std()
z_p = (s - mu_p) / sd_p

al_c = np.where(z_c.abs() > TH)[0]
al_p = np.where(z_p.abs() > TH)[0]
print(f"Janela CENTRADA : {len(al_c)} alarmes")
print(f"Janela CAUSAL   : {len(al_p)} alarmes · primeiro em t = {al_p[0]} "
      f"(a falha começa em t = {T_FALHA})")
print(f"Falsos alarmes antes da falha: centrada {int((al_c < T_FALHA).sum())} · "
      f"causal {int((al_p < T_FALHA).sum())}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7.2), sharex=True,
                         gridspec_kw={"height_ratios": [1.5, 1, 1]})

axes[0].plot(s.index, s.values, color=CINZA, lw=0.9, label="leitura do sensor")
axes[0].plot(mu_c.index, mu_c.values, color=VERM, lw=2, label=f"média centrada (W={W})")
axes[0].plot(mu_p.index, mu_p.values, color=AZUL, lw=2, label=f"média causal (W={W})")
axes[0].axvline(T_FALHA, color="black", ls="--", lw=1.2)
axes[0].text(T_FALHA + 6, s.min() + 0.3, "falha real começa aqui", fontsize=9)
axes[0].set_ylabel("valor"); axes[0].legend(fontsize=9, ncol=3)
axes[0].set_title("A média centrada 'aprende' a falha antes de ela acontecer")

for ax, z, al, cor, nome in [(axes[1], z_c, al_c, VERM, "centrada (usa o futuro)"),
                             (axes[2], z_p, al_p, AZUL, "causal (só o passado)")]:
    ax.plot(z.index, z.abs().values, color=cor, lw=1.1)
    ax.axhline(TH, color=CINZA, ls="--", lw=1.2)
    ax.axvline(T_FALHA, color="black", ls="--", lw=1.2)
    ax.scatter(al, np.abs(z.values)[al], color=cor, s=45, zorder=3)
    ax.set_ylabel("|z|"); ax.set_ylim(0, max(6, np.nanmax(np.abs(z.values)) * 1.1))
    ax.set_title(f"Janela {nome} — {len(al)} alarme(s)", fontsize=11)
axes[2].set_xlabel("t (leituras)")
plt.tight_layout(); plt.show()

### Animação 3 — a janela deslizando

À esquerda a janela centrada, à direita a causal. Acompanhe o instante em que a
falha entra em cena: a janela vermelha já contém as leituras pós-falha **antes**
de `t = 380`, então nada parece anômalo. A azul não tem essa informação — e dispara.

In [ ]:
def animar_janelas(s, W=41, t_falha=380, ini=300, fim=430, passo=2):
    n = len(s)
    frames = list(range(ini, fim, passo))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    faixas, marcas, linhas_t = [], [], []

    for ax, cor, nome in [(axes[0], VERM, "centrada — enxerga o futuro"),
                          (axes[1], AZUL, "causal — só o passado")]:
        ax.plot(range(n), s.values, color=CINZA, lw=0.8)
        ax.axvline(t_falha, color="black", ls="--", lw=1.2)
        faixas.append(ax.axvspan(0, 1, color=cor, alpha=0.20))
        marcas.append(ax.plot([], [], "o", color=cor, ms=9, zorder=4)[0])
        linhas_t.append(ax.plot([], [], color=cor, lw=2)[0])
        ax.set_xlim(ini - W, fim + W); ax.set_xlabel("t"); ax.set_title(nome, fontsize=11)
    axes[0].set_ylabel("valor do sensor")
    sup = fig.suptitle("", color=AZUL, fontweight="bold")

    def frame(k):
        t = frames[k]
        jan = [(t - W // 2, t + W // 2), (t - W, t - 1)]      # centrada, causal
        for i, (a, b) in enumerate(jan):
            a, b = max(0, a), min(n - 1, b)
            faixas[i].set_x(a); faixas[i].set_width(max(b - a, 1))
            m = s.values[a:b + 1].mean()
            linhas_t[i].set_data([a, b], [m, m])
            marcas[i].set_data([t], [s.values[t]])
        estado = "a janela JÁ contém a falha" if t + W // 2 >= t_falha and t < t_falha else ""
        sup.set_text(f"t = {t}   {estado}")
        return faixas + marcas + linhas_t + [sup]

    return animation.FuncAnimation(fig, frame, frames=len(frames), interval=110, blit=False)

mostrar(animar_janelas(s, W=W, t_falha=T_FALHA))

> **Como evitar:** em qualquer coisa com eixo temporal — se a informação usada para
> decidir sobre o instante *t* não estava disponível em *t*, o resultado não vale.
> Vale para janela centrada, para preencher vazios com a média geral e para calcular
> estatísticas sobre o período inteiro.

---
## 7 · Fechamento — o padrão por trás dos cinco

> **Quando a métrica fica boa demais, ela normalmente está medindo outra coisa.**

Em todos os cinco casos o número final parecia ótimo justamente porque algo entrou
na conta sem ser convidado: a escala de um atributo, a geometria em vez do negócio,
a classe majoritária, a popularidade do produto, o futuro.

In [ ]:
resumo = pd.DataFrame([
    ["01", "Não padronizar",           "ARI",       f"{ari_bruto:.2f}", f"{ari_pad:.2f}",
     "escala vira peso implícito"],
    ["02", "Gráfico escolhe k",        "k",         f"{k_silhueta}",    f"{k_verdade}",
     "métrica mede geometria, não utilidade"],
    ["03", "Acurácia com alvo raro",   "F1",        f"{sweep.f1.iloc[-1]:.2f}",
     f"{sweep.f1.iloc[0]:.2f}", "acurácia ignora a classe rara"],
    ["04", "Confiança sem lift",       "lift",      f"{r1['lift']:.2f}", f"{r2['lift']:.2f}",
     "popularidade disfarçada de relação"],
    ["05", "Usar o futuro",            "alarmes",   f"{len(al_c)}",      f"{len(al_p)}",
     "informação indisponível em t"],
], columns=["#", "erro", "métrica", "versão ingênua", "versão correta", "o que entrou na conta"])
display(resumo)

In [ ]:
# checklist operacional — rode isto antes de apresentar qualquer resultado não supervisionado
checklist = [
    "Padronizei? E conferi se todo atributo merece o mesmo peso?",
    "O k saiu da conversa com a operação, não só do gráfico?",
    "Em base desbalanceada, reportei precisão e recall — e o custo dos alertas?",
    "Toda regra de associação foi comparada contra a popularidade do item sozinho?",
    "Toda estatística usa apenas informação disponível no instante da decisão?",
    "Consigo contar a história do resultado para alguém do negócio conferir?",
]
for i, item in enumerate(checklist, 1):
    print(f"  [ ] {i}. {item}")

---
## 8 · Atividade para a próxima aula

Escolha uma base pública e aplique um agrupamento simples. O objetivo não é acertar
o algoritmo — é praticar a desconfiança.

Sugestões: *Online Retail II* (UCI) · *Mall Customer Segmentation* (Kaggle) · *Iris*.

1. **Rode duas vezes** — com e sem padronizar. Descreva o que mudou.
2. **Escolha o número de grupos** e escreva por que escolheu esse — sem citar só o gráfico.
3. **Dê nome aos grupos** — um nome que alguém de fora da área entenda.
4. **Escreva um parágrafo** começando com: *"Este resultado quebraria se…"*

> O item 2 é o que vale a nota: é ali que se vê quem entendeu que a decisão é sua,
> não do gráfico.

---

### Para ir além

- **LUDERMIR, T. B. (2021).** Inteligência Artificial e Aprendizado de Máquina: estado
  atual e tendências. *Estudos Avançados*, v. 35, n. 101, p. 85-94.
- **SUAIDE, A. A. P. (2025).** Introdução a métodos de aprendizado de máquina não
  supervisionados através de um experimento simples. *Rev. Bras. Ensino Fís.*, v. 47, e20240443.
- **GUI, J. et al. (2023).** A Survey on Self-supervised Learning. arXiv:2301.05712.
- **BISHOP, C. M. (2006).** *Pattern Recognition and Machine Learning*. Springer — cap. 9 e 12.

**Glossário:** *clustering* = agrupar por similaridade · *anomalia* = o que não pertence ·
*lift* = ganho sobre o acaso · *look-ahead bias* = usar informação do futuro.